# Imports

In [1]:
import logging, warnings; logging.getLogger().setLevel(logging.ERROR);
warnings.filterwarnings("ignore")

import scanpy as sc
import scanpy.external as sce
import numpy as np
import pandas as pd
import re
from pathlib import Path 
import os

import warnings, scipy.sparse as sp, matplotlib, matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.pyplot import rc_context
import matplotlib.font_manager
import matplotlib.lines as lines


pd.set_option('display.max_rows', 200)

matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype'] = 42
matplotlib.rcParams['font.family'] = 'sans-serif'
matplotlib.rcParams['font.sans-serif'] = 'Arial'
matplotlib.rc('font', size=12)

sc.settings.n_jobs=-1
sc.set_figure_params(dpi=80, dpi_save=300, color_map='Spectral_r', vector_friendly=True, transparent=True)
sc.settings.figdir = '../../../1_outputs/0_figures'
sc.settings.verbosity = 1 # verbosity: errors (0), warnings (1), info (2), hints (3)
sc.logging.print_header()

%matplotlib inline 
%config InlineBackend.figure_format = 'retina'

In [2]:
# preset color palettes and color maps
user_defined_palette =  [ '#F6222E', '#16FF32', '#3283FE', '#FEAF16', '#BDCDFF', '#3B00FB', '#1CFFCE', '#C075A6', '#F8A19F', '#B5EFB5', '#FBE426', '#C4451C', 
                          '#2ED9FF', '#c1c119', '#8b0000', '#FE00FA', '#1CBE4F', '#1C8356', '#0e452b', '#AA0DFE', '#B5EFB5', '#325A9B', '#90AD1C']

user_defined_cmap_markers = LinearSegmentedColormap.from_list('mycmap', ["#E6E6FF", "#CCCCFF", "#B2B2FF", "#9999FF",  "#6666FF",   "#3333FF", "#0000FF"])
user_defined_cmap_degs = LinearSegmentedColormap.from_list('mycmap', ["#0000FF", "#3333FF", "#6666FF", "#9999FF", "#B2B2FF", "#CCCCFF", "#E6E6FF", "#E6FFE6", "#CCFFCC", "#B2FFB2", "#99FF99", "#66FF66", "#33FF33", "#00FF00"])

In [17]:
def sanitize_sheet_name(name):
    return name.replace(':', '_').replace('/', '_')


In [4]:
pwd

'/Users/mkaur/github/2_tff1/1_tecs/1_pyzone/0_notebooks/0_scRNA/2_deg'

In [5]:
deg_outputs = '/Users/mkaur/github/2_tff1/1_tecs/1_pyzone/1_outputs/1_degs/' 
h5ad = '/Users/mkaur/github/2_tff1/1_tecs/1_pyzone/3_h5ad/'

In [6]:
## read in h5ad 

adata = sc.read_h5ad(h5ad + "8_adata_final.h5ad")
adata


AnnData object with n_obs × n_vars = 10336 × 33594
    obs: 'sample', 'log10_original_total_counts', 'n_genes_by_counts', 'ribo_frac', 'mito_frac', 'cell_type', 'cell_type_subset'
    uns: 'cell_type_colors', 'cell_type_subset_colors', 'log1p', 'neighbors', 'pca', 'sample_colors', 'umap'
    obsm: 'X_pca', 'X_umap'
    varm: 'PCs'
    layers: 'raw_data'
    obsp: 'connectivities', 'distances'

In [19]:
adata.obs['cell_type_subset'].value_counts()

cell_type_subset
0:cTEC                        2413
11:mimetic(tuft)              1838
3:mTEC2                       1231
5:mimetic(corneocyte-like)    1209
4:mimetic(goblet)              783
1:mTEC1                        498
7:mimetic(neuroendo)           406
14:vEC                         391
13:capEC                       371
19:MEC                         246
6:mimetic(microfold)           222
2:mTEC-prol                    168
18:vSMC/PC                     113
16:capsFB/intFB                112
12:aEC                         110
9:mimetic(ciliated)             73
15:fetalEC                      57
8:mimetic(ionocyte)             52
17:medFB                        22
10:mimetic(parathyroid)         21
Name: count, dtype: int64

In [9]:
sorted(adata.obs['cell_type_subset'].unique().to_list())

['0:cTEC',
 '10:mimetic(parathyroid)',
 '11:mimetic(tuft)',
 '12:aEC',
 '13:capEC',
 '14:vEC',
 '15:fetalEC',
 '16:capsFB/intFB',
 '17:medFB',
 '18:vSMC/PC',
 '19:MEC',
 '1:mTEC1',
 '2:mTEC-prol',
 '3:mTEC2',
 '4:mimetic(goblet)',
 '5:mimetic(corneocyte-like)',
 '6:mimetic(microfold)',
 '7:mimetic(neuroendo)',
 '8:mimetic(ionocyte)',
 '9:mimetic(ciliated)']

In [10]:
writer = pd.ExcelWriter(deg_outputs + "ko_vs_wt_by_celltype.xlsx", engine="xlsxwriter")

In [11]:
cell_types = adata.obs["cell_type_subset"].unique().tolist()   # change key if needes

In [18]:
for ct in cell_types:
    ad = adata[adata.obs["cell_type_subset"] == ct].copy()

    if ad.obs["sample"].nunique() < 2:
        continue

    sc.tl.rank_genes_groups(
        ad,
        groupby="sample",
        groups=["ko"],
        reference="wt",
        method="wilcoxon",
        use_raw=False
    )

    res = ad.uns["rank_genes_groups"]

    df = pd.DataFrame({
        "gene": res["names"]["ko"],
        "score": res["scores"]["ko"],
        "logFC": res["logfoldchanges"]["ko"],
        "pval_adj": res["pvals_adj"]["ko"],
    })

    df["Is_Significant"] = (df["logFC"].abs() > 0.585) & (df["pval_adj"] < 0.05)

    df.to_excel(
        writer,
        sheet_name=sanitize_sheet_name(ct)[:31],
        index=False
    )

writer.close()